In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

warnings.filterwarnings("ignore")
import os
import pickle
import numpy as np
import pandas as pd
from utils.moodle_connection import moodle_connection
from utils.neo4j_connection import neo4j_connection

# MOODLE credentials
moodle_settings = {
    "host": "...",
    "user": "...",
    "password": "...",
    "port": ...,
    "database": "...",
}
# Neo4j credentials
neo4j_settings = {
    "connection_url": "...",
    "username": "...",
    "password": "...",
}

# Connection with Neo4j and SQL server
graph = neo4j_connection(neo4j_settings=neo4j_settings, clean_graph=False)
connection = moodle_connection(moodle_settings)
cursor = connection.cursor()


metrics = [
    "Cognitive",
    "Collaboration",
    "Critical thinking",
    "Communication",
    "Creativity",
]
convert_clusters_to_profiles = {0: "A", 1: "B", 2: "C"}
path = "./models/Demo/"
img_path = "./images/Demo"

if not os.path.exists(path):
    os.makedirs(path, exist_ok=True)
if not os.path.exists(img_path):
    os.makedirs(img_path, exist_ok=True)

### Data retrieval

In [ ]:
from utils.profiles import retrieve_data_for_profile_assignment

course_id = ...

d_questions, user_responses, d_metrics = retrieve_data_for_profile_assignment(
    course_id=course_id,
    pilot="Demo",
    graph=graph,
    cursor=cursor,
    moodle_settings=moodle_settings,
)

### Create features

In [ ]:
records = list()
for user_id in user_responses:
    # Sanity check
    if user_id not in d_metrics:
        continue

    # Get questionnaire responses
    record = user_responses[user_id].copy()

    # Include engagement metrics
    record.append(d_metrics[user_id]["Autonomy"])
    record.append(d_metrics[user_id]["Competence"])
    record.append(d_metrics[user_id]["Relatedness"])

    # Add all grades per metric
    for metric in metrics:
        if metric not in d_metrics[user_id]:
            record.append(np.nan)
        else:
            record.append(d_metrics[user_id][metric])
    # Include instance
    records.append(record)


# Records (questionnaire + grades)
records = np.asarray(records)
# Feature names (in Greek)
features = (
    [items["question"] for _, items in d_questions.items()]
    + ["Degree of autonomy", "Degree of competence", "Degree of connectivity"]
    + [
        "Degree of cognitive skills",
        "Degree of collaboration",
        "Degree of critical thinking",
        "Degree of communication",
        "Degree of creativity",
    ]
)


# Imputation
# -------------------------------------------------------------------------
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=3, weights="uniform")
# Train and apply imputer
records = imputer.fit_transform(records)
# Save imputer
with open(f"{path}/imputer.pkl", "wb") as handle:
    pickle.dump(imputer, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Clustering

### Preprocessing

In [ ]:
import umap
from sklearn.preprocessing import RobustScaler
from utils.clustering_utils import evaluate_clusters

# Data scaling
scaler = RobustScaler()
data = scaler.fit_transform(records)

# Dimensionality reduction
umap_model = umap.UMAP(
    n_neighbors=...,
    n_components=...,
    metric=...,
    random_state=42,
    n_jobs=-1,
    verbose=False,
)

# Data in 2D
data_2d = umap_model.fit_transform(data)
# Available colors
import matplotlib.pyplot as plt
plt.scatter(x=data_2d[:, 0], y=data_2d[:, 1], s=10, marker="o")

### k-Means++

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Identify the number of clusters value
print("#Clusters\tSilhouette\tCalinski-Harabasz\tDavies-Bouldin\t   DBCV")
for n_clusters in (K := range(2, 11)):
    # Setup K-Means model
    model = KMeans(
        n_clusters=n_clusters, init="k-means++", random_state=42, n_init="auto"
    )

    # Fit model
    model.fit(data_2d)

    # Calculate scores
    scores = evaluate_clusters(data_2d, model.labels_)
    print(
        f"{n_clusters:5.0f}\t\t {scores['Silhouette_score']:+.5f}\t {scores['Calinski_Harabasz_score']:8.2f}\t\t   {scores['Davies_Bouldin_score']:7.5f}\t {scores['DBCV']:+.3f}"
    )

# Select number of clusters
n_clusters = ...

# Create clustering model
model = KMeans(n_clusters=n_clusters, init="k-means++", random_state=42, n_init="auto")
model.fit(data_2d)

# Get labels and centroids
labels = model.labels_
centroids = model.cluster_centers_

#### Visualization

In [ ]:
# Available colors
colors = ["b", "r", "g", "c", "k", "m", "y"]

for i, color in zip(range(n_clusters), colors):
    idx = np.where(labels == i)
    plt.scatter(x=data_2d[idx, 0], y=data_2d[idx, 1], s=10, marker="o", color=color)
    plt.scatter(x=centroids[i, 0], y=centroids[i, 1], s=50, marker="^", color=color)

plt.figure(figsize=(6, 2))
plt.hist(labels);

#### Qualitative cluster evaluation

In [ ]:
import textwrap

figsize = (10, 2)
image_idx = 0
# Categorical features
for question_id, question in enumerate(features[:-8]):
    plt.figure(figsize=figsize)

    # Create histogram data
    bins = np.linspace(0, 5, 6)
    width = (bins[1] - bins[0]) / (n_clusters + 1)
    for i in range(0, n_clusters):
        idx = np.where(labels == i)[0]
        hist, bins = np.histogram(records[idx, question_id], bins=bins)
        plt.bar(
            bins[:-1] + i * width - width / n_clusters,
            hist,
            width=width,
            label=f"{convert_clusters_to_profiles[i]}",
            align="center",
            alpha=0.5,
        )

    plt.xticks(
        list(d_questions[question_id]["choices"]),
        list(d_questions[question_id]["choices"].values()),
        rotation=10,
    )
    wrapped_title = "\n".join(textwrap.wrap(question, width=100))
    plt.legend(frameon=False)
    plt.title(wrapped_title, fontsize=10)
    plt.savefig(f"{img_path}/{image_idx}.png", dpi=300, bbox_inches="tight")
    image_idx += 1
    plt.show()


# Engagement metrics
for engagement_id, engagement_metric in zip(
    [
        -8,
        -7,
        -6,
    ],
    ["Autonomy", "Competence", "Relatedness"],
):
    plt.figure(figsize=figsize)

    # Create histogram data
    bins = np.linspace(0, 5, 6)
    width = (bins[1] - bins[0]) / (n_clusters + 1)
    for i in range(0, n_clusters):
        idx = np.where(labels == i)[0]
        hist, bins = np.histogram(records[idx, engagement_id], bins=bins)
        plt.bar(
            bins[:-1] + i * width - width / n_clusters,
            hist,
            width=width,
            label=f"{convert_clusters_to_profiles[i]}",
            align="center",
            alpha=0.5,
        )

    plt.xticks([0, 1, 2, 3, 4], ["Low", "Medium", "High", "Very high", ""], rotation=10)
    wrapped_title = "\n".join(textwrap.wrap(engagement_metric, width=100))
    plt.legend(frameon=False)
    plt.title(wrapped_title, fontsize=10)
    plt.savefig(f"{img_path}/{image_idx}.png", dpi=300, bbox_inches="tight")
    image_idx += 1
    plt.show()

# Real features (grades)
for idx in [i for i in range(-len(metrics), 0, 1)]:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=figsize)
    ax.boxplot([records[np.where(labels == i)[0], idx] for i in range(0, n_clusters)])
    plt.title(features[idx], fontsize=10)
    plt.xticks(
        [i + 1 for i in range(n_clusters)],
        [convert_clusters_to_profiles[i] for i in range(n_clusters)],
        rotation=10,
    )
    plt.savefig(f"{img_path}/{image_idx}.png", dpi=300, bbox_inches="tight")
    image_idx += 1
    plt.show()

### GMM

In [ ]:
from sklearn.mixture import GaussianMixture

print("#Clusters\tSilhouette\tCalinski-Harabasz\tDavies-Bouldin\t   DBCV")
for n_clusters in (K := range(2, 11)):
    # Setup GaussianMixture model
    model = GaussianMixture(n_components=n_clusters, random_state=42)

    # Fit model
    model.fit(data_2d)

    # Calculate scores
    scores = evaluate_clusters(data_2d, model.predict(data_2d))
    print(
        f"{n_clusters:5.0f}\t\t {scores['Silhouette_score']:+.5f}\t {scores['Calinski_Harabasz_score']:8.2f}\t\t   {scores['Davies_Bouldin_score']:7.5f}\t {scores['DBCV']:+.3f}"
    )


# Select number of clusters
n_clusters = ...

# Create clustering model
model = GaussianMixture(n_components=n_clusters, random_state=42)
model.fit(data_2d)

# Get labels and centroids
labels = model.predict(data_2d)
centroids = model.means_

#### Visualization

In [ ]:
# Available colors
colors = ["b", "r", "g", "c", "k", "m", "y"]

for i, color in zip(range(n_clusters), colors):
    idx = np.where(labels == i)
    plt.scatter(x=data_2d[idx, 0], y=data_2d[idx, 1], s=10, marker="o", color=color)
    plt.scatter(x=centroids[i, 0], y=centroids[i, 1], s=50, marker="^", color=color)

plt.figure(figsize=(6, 2))
plt.hist(labels);

#### Qualitative cluster evaluation

In [ ]:
import textwrap

figsize = (10, 2)
image_idx = 0
# Categorical features
for question_id, question in enumerate(features[:-8]):
    plt.figure(figsize=figsize)

    # Create histogram data
    bins = np.linspace(0, 5, 6)
    width = (bins[1] - bins[0]) / (n_clusters + 1)
    for i in range(0, n_clusters):
        idx = np.where(labels == i)[0]
        hist, bins = np.histogram(records[idx, question_id], bins=bins)
        plt.bar(
            bins[:-1] + i * width - width / n_clusters,
            hist,
            width=width,
            label=f"{convert_clusters_to_profiles[i]}",
            align="center",
            alpha=0.5,
        )

    plt.xticks(
        list(d_questions[question_id]["choices"]),
        list(d_questions[question_id]["choices"].values()),
        rotation=10,
    )
    wrapped_title = "\n".join(textwrap.wrap(question, width=100))
    plt.legend(frameon=False)
    plt.title(wrapped_title, fontsize=10)
    plt.savefig(f"{img_path}/{image_idx}.png", dpi=300, bbox_inches="tight")
    image_idx += 1
    plt.show()


# Engagement metrics
for engagement_id, engagement_metric in zip(
    [
        -8,
        -7,
        -6,
    ],
    ["Autonomy", "Competence", "Relatedness"],
):
    plt.figure(figsize=figsize)

    # Create histogram data
    bins = np.linspace(0, 5, 6)
    width = (bins[1] - bins[0]) / (n_clusters + 1)
    for i in range(0, n_clusters):
        idx = np.where(labels == i)[0]
        hist, bins = np.histogram(records[idx, engagement_id], bins=bins)
        plt.bar(
            bins[:-1] + i * width - width / n_clusters,
            hist,
            width=width,
            label=f"{convert_clusters_to_profiles[i]}",
            align="center",
            alpha=0.5,
        )

    plt.xticks([0, 1, 2, 3, 4], ["Low", "Medium", "High", "Very high", ""], rotation=10)
    wrapped_title = "\n".join(textwrap.wrap(engagement_metric, width=100))
    plt.legend(frameon=False)
    plt.title(wrapped_title, fontsize=10)
    plt.savefig(f"{img_path}/{image_idx}.png", dpi=300, bbox_inches="tight")
    image_idx += 1
    plt.show()

# Real features (grades)
for idx in [i for i in range(-len(metrics), 0, 1)]:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=figsize)
    ax.boxplot([records[np.where(labels == i)[0], idx] for i in range(0, n_clusters)])
    plt.title(features[idx], fontsize=10)
    plt.xticks(
        [i + 1 for i in range(n_clusters)],
        [convert_clusters_to_profiles[i] for i in range(n_clusters)],
        rotation=10,
    )
    plt.savefig(f"{img_path}/{image_idx}.png", dpi=300, bbox_inches="tight")
    image_idx += 1
    plt.show()

## Create Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# The parameters are suggested from the performance evaluation conducted
model = RandomForestClassifier(
    n_estimators=30, n_jobs=-1, max_depth=3, max_features="log2"
)
model.fit(X=records, y=labels)
with open(f"{path}/....pkl", "wb") as handle:
    pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Get predictions
pred = model.predict(records)
# Classification report
from sklearn.metrics import classification_report
print(classification_report(labels, pred))

In [ ]:
plt.figure(figsize=(5, 5))

for i, color in zip(range(n_clusters), colors):
    idx = np.where(pred == i)
    plt.scatter(x=data_2d[idx, 0], y=data_2d[idx, 1], s=10, marker="o", color=color)
    plt.scatter(x=centroids[i, 0], y=centroids[i, 1], s=50, marker="^", color=color)

#### Create explainer

In [ ]:
# LIME has one explainer for all models
categorical_names = {key: items["choices"] for key, items in d_questions.items()}
for i in range(0, 3):
    categorical_names[len(categorical_names)] = {
        0: "Low",
        1: "Medium",
        2: "High",
        3: "Very high",
    }

from lime import lime_tabular
explainer = lime_tabular.LimeTabularExplainer(
    training_data=records,
    feature_names=features,
    categorical_features=range(0, len(features[: -len(metrics)])),
    categorical_names=categorical_names,
    class_names=list(map(str, range(n_clusters))),
    mode="classification",
)

#### Test explanator

In [ ]:
idx = 17

new_instance = records[idx].copy()
print("augMENTOR profile: ", model.predict(new_instance.reshape(1, -1)))
print(
    "Prediction probability: ",
    100 * np.max(model.predict_proba(new_instance.reshape(1, -1))),
)

exp = explainer.explain_instance(
    data_row=new_instance, num_features=len(features), predict_fn=model.predict_proba
)

exp.show_in_notebook(show_table=True)

In [ ]:
explanation = ""

for item, score in exp.as_list():
    if score < 0.0:
        continue

    if "<" in item or ">" in item:
        explanation += f"- {item}\n"
    else:
        question, answer = item.split("=")
        explanation += f"- '{question}' = '{answer}'\n"

    # if explanation.count("\n") == 5: break
print(explanation)


In [ ]:
import json
from langchain.chat_models import ChatOpenAI
from langchain.prompts.prompt import PromptTemplate
from langchain.chains.llm import LLMChain
from utils.prompts import prompt_template

llm = ChatOpenAI(model_name="...", max_tokens=..., temperature=0.0, openai_api_key="") # add your key
prompt = PromptTemplate(input_variables=['profile', 'profile_description', 'probability', 'LIME'], template=prompt_template)
chain = LLMChain(llm=llm, prompt=prompt)

with open("Data/profiles_description.json", "r", encoding="utf-8") as file:
    d_profiles_description = json.load(file)

profile = convert_clusters_to_profiles[model.predict(new_instance.reshape(1, -1))[0]]
probability = 100 * np.max(model.predict_proba(new_instance.reshape(1, -1)))
explanation = chain.run({'profile': profile, 
                    'profile_description': d_profiles_description['Demo'][profile]['description'], 
                    'probability': str(probability), 
                    'LIME': explanation})
